In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# RotationRenderer：小型内容对照

**不要直接点击“全部运行”。** 本包须经控制会话批准后手动执行。先运行到六路线 pilot，查看真实耗时与完整性，再手动继续剩余46条。

固定旧R0前两对CG/G，±15°、黑色/反射、双线性，分别测攻击图pre、预测H校正post、oracle；加四张未攻击参考，共52条计划路线。G有同步水印、无内容水印，不代表完全无水印FPR。主参考tau=1.2657276026437319，仅描述；完整m是核心。

复用旧图，不生成或重新嵌入；CG的8份几何来自已审64行CPU诊断，随固定代码携带。G在本包独立检测8次。无V2搜索或评分优化，旧正式结果保留。

## 1. 固定代码与原始 Drive 输入
只复制原R0 result及四张CG/G到Colab临时cache，不修改或上传原Drive资料。

In [ ]:
from pathlib import Path
import json, sys, subprocess, shutil, time
EXACT = '878488548dded46f3a5e90b481a968774f2afbe8'
REPO = Path('/content/ceg-wm-renderer-content-8784885')
R0 = Path('/content/drive/MyDrive/CEG-WM/Geometry-V7/4f0bf1560805672f786dc86dd50d793aec18aae7/r0-f1')
CACHE = Path('/content/renderer-content-inputs-8784885')
OUTPUT = Path('/content/drive/MyDrive/CEG-WM/RotationRenderer-Diagnostic-V1/content-renderer-v1')
if OUTPUT.exists():
    raise FileExistsError('已有内容输出，保留原结果，不自动覆盖或重跑。')
if not REPO.exists():
    subprocess.run(['git','clone','--branch','RotationRenderer-Diagnostic-V1','--single-branch','https://github.com/RICHAAARC/CEG-WM.git',str(REPO)],check=True)
elif subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
    raise RuntimeError('保留已有代码改动。')
subprocess.run(['git','-C',str(REPO),'checkout','--detach',EXACT],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q',str(REPO)],check=True)
sys.path.insert(0,str(REPO))
sys.path.insert(0,str(REPO/'src'))
from diagnostics.rotation_renderer.content_plan import ContentSession, prepare_sources, validate_scoring_assets, UNITS, ARMS, ARM_NAMES
REFERENCE = REPO/'diagnostics/rotation_renderer/cg_geometry_reference.json'
original = json.loads((R0/'result.json').read_text(encoding='utf-8-sig'))
mapping = []
for unit in UNITS:
    records = [r for r in original['raw_unit_records'] if r['stage']=='evaluation' and r['unit_id']==unit]
    if len(records)!=1: raise ValueError('原R0单元缺失或重复')
    for arm in ARMS:
        arms = [r for r in records[0]['arms'] if r['arm']==ARM_NAMES[arm]]
        if len(arms)!=1: raise ValueError('原R0 arm缺失或重复')
        relative = arms[0]['image_file']
        source = (R0/relative).resolve()
        if not source.is_relative_to(R0.resolve()) or not source.is_file():
            raise FileNotFoundError('原CG/G路径不存在或不在R0目录内')
        mapping.append(dict(unit_id=unit,arm=arm,original_path=relative,local_filename=f'{unit}__{arm}.png'))
CACHE.mkdir(parents=True,exist_ok=False)
shutil.copyfile(R0/'result.json',CACHE/'r0-result.json')
for item in mapping:
    shutil.copyfile(R0/item['original_path'],CACHE/item['local_filename'])
(CACHE/'content_source_mapping.json').write_text(json.dumps({'images':mapping},indent=2))
renderer_r0, renderer_sources = prepare_sources(CACHE)
print({'code':EXACT,'original_pairs':len(renderer_sources)//2,'planned_routes':52,'output':str(OUTPUT)})

## 2. 原密钥与生产资产初始化

密钥来自Colab Secret `CEG_WM_ROOT_KEY`，与原R0 key身份核对但不输出原文。优先复用当前运行时已有且类型/计算资产匹配的生产assets；否则读取`HF_TOKEN`并使用既有production factory。不会另造优化loader。新建factory会加载SD3.5及DINO，初始化时间单列；不限定A100。

In [ ]:
from google.colab import userdata
import torch
try:
    renderer_key = userdata.get('CEG_WM_ROOT_KEY')
except Exception:
    raise RuntimeError('请启用Colab Secret CEG_WM_ROOT_KEY访问') from None
if not renderer_key: raise RuntimeError('CEG_WM_ROOT_KEY为空')
from cegwm.shared.keys import public_key_digest
if public_key_digest(renderer_key) != renderer_r0['public_key_digest']:
    raise ValueError('Secret与原R0嵌入密钥不一致；先修正密钥，尚未初始化模型。')
setup_started = time.perf_counter()
renderer_assets = None
renderer_reuse = None
candidates = [(name,globals().get(name)) for name in ('production_assets','assets','runtime_assets')]
previous_session = globals().get('session')
if previous_session is not None:
    candidates.append(('session.assets',getattr(previous_session,'assets',None)))
for label,candidate in candidates:
    if candidate is None: continue
    try:
        validate_scoring_assets(candidate,renderer_key,renderer_r0)
    except (TypeError,ValueError,AttributeError):
        continue
    renderer_assets,renderer_reuse = candidate,label
    break
if renderer_assets is None:
    try: renderer_token = userdata.get('HF_TOKEN')
    except Exception: raise RuntimeError('请启用Colab Secret HF_TOKEN访问') from None
    if not renderer_token: raise RuntimeError('HF_TOKEN为空')
    from experiments.run_blind_detection_v1 import build_production_runtime, load_runtime_config
    runtime_root = CACHE/'runtime'
    runtime_root.mkdir(exist_ok=False)
    renderer_pipeline,renderer_assets = build_production_runtime(REPO,load_runtime_config(REPO),hf_token=renderer_token,runtime_root=runtime_root)
    del renderer_token
    renderer_reuse = 'fresh existing production factory'
renderer_session = ContentSession(CACHE,REFERENCE,OUTPUT,renderer_key,renderer_assets)
setup = {'initialization_seconds':time.perf_counter()-setup_started,'asset_source':renderer_reuse,
         'torch':torch.__version__,'cuda':torch.cuda.is_available(),
         'gpu':torch.cuda.get_device_name() if torch.cuda.is_available() else None,
         'geometry_device':str(renderer_assets.geometry_backend.device),'code':EXACT}
(OUTPUT/'initialization.json').write_text(json.dumps(setup,indent=2))
print(setup)

## 3. 先运行六路线 pilot

固定0001、+15°、黑色、CG/G各pre/post/oracle。检查实际调用、完整统计、错误和耗时；计划数不代表完成数。oracle仅诊断，不能当正式盲检测。运行后停在这里查看输出。

In [ ]:
pilot = renderer_session.pilot()
(OUTPUT/'pilot_summary.json').write_text(json.dumps(pilot,indent=2))
print(json.dumps(pilot,indent=2))

## 4. 查看 pilot 后手动继续46条

本单元先记录四张未攻击CG/G参考，再完成剩余攻击路线。pilot分数差或有缺H不能让这些参考从结果中消失；保留每条失败，不补样、不调阈值。不要用“全部运行”跳过pilot查看步骤。

In [ ]:
renderer_session.remaining()
print((OUTPUT/'summary.json').read_text())

## 5. 查看并交回结果

返回`content-renderer-v1`目录供原始审计。先比较完整m与负对照，再判断内容是否恢复；黑色候选与反射stress分列。本小包不估计正式FPR，不修改旧结果。

In [ ]:
import pandas as pd
table = pd.read_csv(OUTPUT/'paired.csv')
display(table)
print({'recorded_routes':len(table),'unique_routes':len(table.drop_duplicates(['unit','arm','condition','fill','route']))})